### BUSINESS CHALLENGE:
# Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

# We will be provided a company name and their primary website.

In [22]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from web_screape_tool import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [23]:
load_dotenv()
def connect_to_ollama(): 
    OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
    return OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")

def chat_with_ollama(messages):
    ollama = connect_to_ollama()
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

In [24]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/google/gemma-4-12B-it',
 '/nvidia/LocateAnything-3B',
 '/unsloth/gemma-4-12b-it-GGUF',
 '/google/gemma-4-12B',
 '/ideogram-ai/ideogram-4-fp8',
 '/models',
 '/spaces/VAST-AI/TripoSplat',
 '/spaces/ideogram-ai/ideogram4',
 '/spaces/nvidia/LocateAnything',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview-2',
 '/spaces/prithivMLmods/Qwen-Image-Edit-2511-LoRAs-Fast',
 '/spaces',
 '/datasets/agents-last-exam/agents-last-exam',
 '/datasets/wikimedia/structured-wikipedia',
 '/datasets/stanford-vision-lab/gpic',
 '/datasets/openbmb/UltraData-SFT-2605',
 '/datasets

In [25]:
#step1: generate system prompt which decide which link is usefull for company broucher
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [26]:
#step2: generate user prompt which is called for each link
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [27]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/google/gemma-4-12B-it
/nvidia/LocateAnything-3B
/unsloth/gemma-4-12b-it-GGUF
/google/gemma-4-12B
/ideogram-ai/ideogram-4-fp8
/models
/spaces/VAST-AI/TripoSplat
/spaces/ideogram-ai/ideogram4
/spaces/nvidia/LocateAnything
/spaces/r3gm/wan2-2-fp8da-aoti-preview-2
/spaces/prithivMLmods/Qwen-Image-Edit-2511-LoRAs-Fast
/spaces
/datasets/agents-last-exam/agents-last-exam
/datasets/w

In [28]:
#step3: select the relevant links for the brochure
load_dotenv()
def connect_to_ollama(): 
    OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
    return OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")

def chat_with_ollama(messages):
    ollama = connect_to_ollama()
    response = ollama.chat.completions.create(model="llama3.2", messages=messages, response_format={"type": "json_object"})
    return response.choices[0].message.content

def select_relevant_links(url):
    message = [
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(url)}
    ]
    response = chat_with_ollama(messages=message)
    result = response
    links = json.loads(result)
    return links

In [29]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'Company page', 'url': 'https://huggingface.co/'},
  {'type': 'About page', 'url': 'https://brand.huggingface.co'},
  {'type': 'Blog page', 'url': 'https://blog.huggingface.co'},
  {'type': 'Pricing page', 'url': 'https://pricing.huggingface.co'},
  {'type': 'Documentation homepage', 'url': 'https://docs.huggingface.co'},
  {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'},
  {'type': 'Discord server',
   'url': 'https://join.discord.com/through-huggingface'},
  {'type': 'Hugging Face Discord discussion forum',
   'url': 'https://discuss.huggingface.co'}]}

In [30]:
# step4: assemble all details in seperate prompt
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [42]:
fetched_pages_and_links =fetch_page_and_all_relevant_links("https://docs.ollama.com/")

In [45]:
print(fetched_pages_and_links)

## Landing Page:

Ollama's documentation - Ollama

Documentation Index
Fetch the complete documentation index at:
/llms.txt
Use this file to discover all available pages before exploring further.
Skip to main content
Ollama
home page
Search...
⌘
K
Get started
Welcome
Quickstart
Cloud
Capabilities
Streaming
Thinking
Structured Outputs
Vision
Embeddings
Tool calling
Web search
Integrations
Overview
Assistants
Coding
IDEs & Editors
Chat & RAG
Automation
Notebooks
More information
CLI Reference
Assistant Sandboxing
Modelfile Reference
Context length
Linux
macOS
Windows
Docker
Importing a Model
FAQ
Hardware support
Troubleshooting
Sign in
Download
Ollama
home page
Search...
⌘
K
Sign in
Download
Download
Search...
Navigation
Get started
Ollama's documentation
Documentation
API Reference
Documentation
API Reference
Get started
Ollama's documentation
Copy page
Copy page
Ollama
is the easiest way to get up and running with large language models such as gpt-oss, Gemma 4, DeepSeek-R1, Qwen3 and m

In [46]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [47]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    # user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt += fetched_pages_and_links
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [49]:
brochure_user_prompt = get_brochure_user_prompt("Ollama", "https://docs.ollama.com/")

In [50]:
def create_brochure(company_name, url):
    response = chat_with_ollama(
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [54]:
# To get the streaming response, we can modify the `chat_with_ollama` function to yield chunks of the response as they come in. Here's how you can do it:
def chat_with_ollama_stream(messages):
    ollama = connect_to_ollama()
    response = ollama.chat.completions.create(model="llama3.2", messages=messages, stream=True)
    for chunk in response:
        yield chunk.choices[0].delta.content

In [58]:
def stream_brochure(company_name, url):
    stream = chat_with_ollama_stream(
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ]
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [60]:
stream_brochure("Ollama", "https://docs.ollama.com/")

**Ollama Brochure**

**Welcome to Ollama**

Ollama is a revolutionary platform designed to make it easy for developers and users to work with large language models. Our mission is to democratize access to cutting-edge natural language processing technologies, empowering individuals and organizations to unlock the full potential of language.

**About Us**

At Ollama, we are passionate about innovation and collaboration. Our team is comprised of experts in AI, linguistics, and software development who share a common vision: to harness the power of language models for the betterment of society. We believe that technology should be accessible to everyone, which is why we design our platform with simplicity, flexibility, and ease of use.

**Our Technology**

Ollama's platform offers an easy-to-use interface to get started with large language models such as gpt-oss, Gemma 4, DeepSeek-R1, Qwen3 and more. Our cloud-based models provide larger models with better performance. We also offer libraries in Python and JavaScript to facilitate seamless integration with popular development tools.

**Our Community**

Join our vibrant community on Discord and Reddit to connect with fellow users, developers, and enthusiasts who share your interest in language models. Stay updated with the latest news, releases, and tutorials through our community-driven channels.

**Careers at Ollama**

If you're passionate about language models and want to join a dynamic team of innovators, check out our current job openings! At Ollama, we offer competitive salaries, comprehensive benefits packages, and opportunities for professional growth.

**Get Started with Ollama Today**

Download our platform on macOS, Windows or Linux to start exploring the possibilities of large language models. With Ollama, you can unlock new opportunities, accelerate innovation, and shape the future of natural language processing.

Stay tuned for updates, releases, and tutorials through our documentation center and social media channels. Join us in revolutionizing the world of language!